In [66]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import trimesh


## Sleeve Skirt, Sleeve Cuff, Collar 등 제거

In [67]:
import os, sys

sys.path.append(os.path.dirname(os.getcwd()))
sys.path.append(os.path.dirname(os.path.dirname(os.getcwd())))

from env_constants import PYGARMENT_ROOT, DATASET_ROOT
sys.path.append(PYGARMENT_ROOT)
import pygarment as pyg





# sys.path.append(PYGARMENT_ROOT)
import pygarment as pyg

import math
import pickle
import numpy as np
import svgpathtools as svgpath
from PIL import Image
from copy import deepcopy
from torch.utils.data import Dataset
import random
import trimesh

from dataclasses import dataclass, field
from typing import List, Dict, Tuple
import numpy as np

import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch



def min_dist_between_edges(edge1, edge2, N_SAMPLE_PER_EDGE = 50) :
    sample_t_arr = np.linspace(0, 1, N_SAMPLE_PER_EDGE)
    edge_sample_point_arr_list = []
    for edge in [edge1, edge2] :
        edge_sample_point_arr_list.append(
            np.array([
                [edge.point(t).real, edge.point(t).imag]
                for t in sample_t_arr
            ])
        )
        
    final_min_dist = np.inf
    for p1 in edge_sample_point_arr_list[0] :
        min_dist = np.min(
            np.linalg.norm(
                p1 - edge_sample_point_arr_list[1],
                axis = 1
            )
        )
        final_min_dist = np.min([final_min_dist, min_dist])
    return final_min_dist


@dataclass
class Stitch :
    panel_0 : str
    edge_0 : int
    panel_1 : str
    edge_1 : int
    stitch_direction : bool
    

class StitchDict:
    def __init__(self, raw_stitch_dict: Dict[int, List[Dict]]):
        """
        Initialize the StitchManager from a raw stitch dictionary.        
        raw_stitch_dict: keys are stitch ids and values are lists of two dictionaries
        with 'panel' and 'edge' keys.
        """
        self._stitches: Dict[int, Stitch] = {}
        for stitch_id, stitch_pair in raw_stitch_dict.items():
            # Each stitch_pair contains two entries representing the two sides of the stitch.
            side0 = stitch_pair[0]
            side1 = stitch_pair[1]
            # Optionally, you could compute stitch_direction from panel names or edges.
            stitch_direction = False  # You can update this logic as needed.
            self._stitches[stitch_id] = Stitch(
                panel_0=side0['panel'],
                edge_0=side0['edge'],
                panel_1=side1['panel'],
                edge_1=side1['edge'],
                stitch_direction=stitch_direction
            )
    
    def reindex(
        self,
        mapping: Dict[int, int],
    ) -> None:
        """
        Re-index the internal dictionary using a provided mapping.
        For example, mapping could be ann_to_img_seam_idx_map or img_to_ann_seam_idx_map.
        """
        new_stitches = {}
        for old_key, stitch in self._stitches.items():
            new_key = mapping.get(old_key, old_key)
            new_stitches[new_key] = stitch
        self._stitches = new_stitches
    
    def __getitem__(self, key: int) -> Stitch:
        return self._stitches[key]
    
    def __setitem__(self, key: int, value: Stitch) -> None:
        self._stitches[key] = value
    
    def items(self):
        return self._stitches.items()

    def __len__(self):
        return len(self._stitches)
    
    def __repr__(self):
        return repr(self._stitches)


class SVGPanel:
    def __init__(self, svg_path: List[svgpath.Path]):
        self.svg_path = svg_path
        
    @property
    def edge_len_list(self) :
        return [edge.length() for edge in self.svg_path]
    
    @property
    def normalized_edge_stt(self, edge_idx : int) :
        return sum(self.edge_len_list[:edge_idx]) / sum(self.edge_len_list)
        
    @property
    def normalized_edge_end(self, edge_idx : int) :
        return sum(self.edge_len_list[:edge_idx+1]) / sum(self.edge_len_list)
    
    def translate(self, dx: float, dy: float) -> None:
        """
        Translate the panel by dx along x-axis and dy along y-axis.
        """
        # In the complex plane, translation by (dx, dy) means adding (dx + dy*j)
        delta = dx + dy * 1j
        self.svg_path = svgpath.Path(*[path.translated(delta) for path in self.svg_path])


    def scale(self, factor: float, pivot: np.ndarray = np.array([0, 0])) -> None:
        """
        Scale the panel by the given factor relative to the pivot.
        By default, the pivot is 0 (the origin), but you can specify a different pivot point.
        """
        pivot_complex = pivot[0] + pivot[1] * 1j
        self.svg_path = svgpath.Path(*[
            (path.translated(-pivot_complex)).scaled(factor).translated(pivot_complex) for path in self.svg_path
        ])
        
    def set_scale_to(self, size : float, use_vert_bbox : bool = False) -> float :
        if use_vert_bbox :
            x1, y1, x2, y2 = self.vert_bbox()
        else :
            x1, y1, x2, y2 = self.bbox()
        width = x2 - x1
        height = y2 - y1

        scale_factor = size / max(width, height)
        self.scale(scale_factor)
        return scale_factor
    
    def set_start_position_at(self, x : float, y : float) -> None :
        start_x = self.svg_path[0].start.real
        start_y = self.svg_path[0].start.imag
        self.translate(x - start_x, y - start_y)
        
    def rotate_clockwise(self, angle_degrees: float, pivot: np.ndarray = np.array([0, 0])) -> None:
        """
        Rotate the entire path by a given angle in degrees around a pivot point.
        
        Args:
            angle_degrees: The angle to rotate the path, in degrees.
            pivot: The point around which to rotate the path. Default is the origin (0+0j).
        """
        pivot_complex = pivot[0] + pivot[1] * 1j
        self.svg_path = svgpath.Path(*[
            path.rotated(angle_degrees, pivot_complex) for path in self.svg_path
        ])
        
    def mirror_horizontal(self) -> None:
        """
        Mirror the panel horizontally (flip the x-axis).
        """
        # self.svg_path = [svgpath.Path(*[self._mirror_horizontal_segment(seg) for seg in path]) for path in self.svg_path]
        self.svg_path = svgpath.Path(*[self._mirror_horizontal_segment(seg) for seg in self.svg_path])        

    def mirror_vertical(self) -> None:
        """
        Mirror the panel vertically (flip the y-axis).
        """
        # self.svg_path = [svgpath.Path(*[self._mirror_vertical_segment(seg) for seg in path]) for path in self.svg_path]
        self.svg_path = svgpath.Path(*[self._mirror_vertical_segment(seg) for seg in self.svg_path])

    def _mirror_horizontal_segment(self, segment):
        if isinstance(segment, svgpath.Arc):
            # Mirror the start and end points horizontally
            start = complex(-segment.start.real, segment.start.imag)
            end = complex(-segment.end.real, segment.end.imag)
            # Reverse the sweep flag
            sweep = not segment.sweep
            return svgpath.Arc(start=start, radius=segment.radius, rotation=segment.rotation,
                       large_arc=segment.large_arc, sweep=sweep, end=end)
        else:
            return segment.scaled(-1, 1)

    def _mirror_vertical_segment(self, segment):
        if isinstance(segment, svgpath.Arc):
            # Mirror the start and end points vertically
            start = complex(segment.start.real, -segment.start.imag)
            end = complex(segment.end.real, -segment.end.imag)
            # Reverse the sweep flag
            sweep = not segment.sweep
            return svgpath.Arc(start=start, radius=segment.radius, rotation=segment.rotation,
                       large_arc=segment.large_arc, sweep=sweep, end=end)
        else:
            return segment.scaled(1, -1)

    def draw(
        self,
        ax          : plt.Axes = None,
        panel_name  : str = None,
        N_SAMPLE_PER_EDGE : int = 80,
        stitch_list : List[int] = None,
        edge_color_list : List[Tuple[float, float, float]] = None
    ) -> None:
        if stitch_list is not None :
            assert len(stitch_list) == len(self.svg_path), "stitch_list must be the same length as the number of edges"
        if ax is None :
            ax = plt.gca()
        if panel_name is not None :
            ax.set_title(panel_name)
        if edge_color_list is None :
            edge_color_list = plt.cm.rainbow(np.linspace(0, 1, len(self.svg_path)))
         
        path_start_pos = np.array([self.svg_path[0].start.real, self.svg_path[0].start.imag])   
        for dx, dy in [(-1, -1), (-1, 1), (1, -1), (1, 1)]:
            ax.annotate(
                "0",
                path_start_pos + np.array([dx, dy]),
                color = "white", fontweight = "bold",
                fontsize = 9
            )
        ax.annotate(
            "0",
            path_start_pos,
            color = "black", # fontweight = "bold",
            fontsize = 9
        )
        for edge_idx, (edge, edge_color) in enumerate(zip(self.svg_path, edge_color_list)) :
            ax.add_patch(
                FancyArrowPatch(
                    [edge.start.real, edge.start.imag],
                    [edge.end.real, edge.end.imag],
                    arrowstyle='-|>',
                    mutation_scale=15,
                    color = "black",
                    linewidth = 0.5
                )
            )
            ax.scatter(
                list(map(lambda t : edge.point(t).real, np.linspace(0, 1, N_SAMPLE_PER_EDGE))),
                list(map(lambda t : edge.point(t).imag, np.linspace(0, 1, N_SAMPLE_PER_EDGE))),
                s = 0.5, color = edge_color
            )
            if stitch_list is not None :
                # for dx, dy in [(-1, -1), (-1, 1), (1, -1), (1, 1)]:
                #     ax.annotate(
                #         f"{stitch_list[edge_idx]}",
                #         [
                #             (edge.start.real + edge.end.real) / 2 + dx,
                #             (edge.start.imag + edge.end.imag) / 2 + dy
                #         ],
                #         color = "white", # fontweight = "bold",
                #         fontsize = 11, fontweight = "bold"
                #     )
                ax.annotate(
                    f"{stitch_list[edge_idx]}",
                    [
                        (edge.start.real + edge.end.real) / 2,
                        (edge.start.imag + edge.end.imag) / 2
                    ],
                    color = edge_color,
                    fontsize = 11, fontweight = "bold"
                )
        ax.invert_yaxis()
        ax.axis("equal")
        
    def is_clockwise(self, n_sample_per_edge : int = 10) -> bool:
        """
        Determine if the path is clockwise.
        """
        total = 0
        for segment in self.svg_path:
            t_list = np.linspace(0, 1, n_sample_per_edge)
            sampled_points = np.array(list(map(
                lambda t : [segment.point(t).real, segment.point(t).imag],
                t_list
            )))
            
            total += np.sum(
                (
                    sampled_points[1:, 0] - sampled_points[:-1, 0]
                ) * (
                    sampled_points[1:, 1] + sampled_points[:-1, 1]
                )
            )
            
            # start = segment.start
            # end = segment.end
            # total += (end.real - start.real) * (end.imag + start.imag)
        return total <= 0


    def reverse_path(self) -> None:
        """
        Reverse the order of the path segments and their directions.
        """
        reversed_segments = []
        for segment in reversed(self.svg_path):
            reversed_segment = segment.reversed()
            reversed_segments.append(reversed_segment)
        self.svg_path = svgpath.Path(*reversed_segments)
        
    def set_start(self, idx: int) -> None :
        """
        Set the start point of the path to the point at index idx.
        """
        self.svg_path = svgpath.Path(*(self.svg_path[idx:] + self.svg_path[:idx]))

        
    def find_narrowest_distance(self, N_SAMPLE_PER_EDGE = 50) :
        edge_idx_combination_list = []
        for edge_idx1 in range(len(list(self.svg_path))) :
            for edge_idx2 in range(edge_idx1 + 1, len(list(self.svg_path))) :
                edge_idx_combination_list.append(
                    (edge_idx1, edge_idx2)
                )
        edge_combination_list = []
        for edge_idx1, edge_idx2 in edge_idx_combination_list :
            if edge_idx1 == edge_idx2 :
                continue
            if np.abs(edge_idx1 - edge_idx2) in [1, len(list(self.svg_path)) - 1] :
                continue
            else :
                edge_combination_list.append(
                    (list(self.svg_path)[edge_idx1], list(self.svg_path)[edge_idx2])
                )
        final_min_dist = np.inf
        for edge1, edge2 in edge_combination_list :
            min_dist = min_dist_between_edges(edge1, edge2, N_SAMPLE_PER_EDGE)
            final_min_dist = np.min([final_min_dist, min_dist])
        return final_min_dist
    
    def bbox(self) :
        """
        (xmin, ymin, xmax, ymax)
        """
        xmin, xmax, ymin, ymax = svgpath.Path(*self.svg_path).bbox()
        return xmin, ymin, xmax, ymax
    
    def vert_bbox(self) :
        vert_list = []
        for edge in self.svg_path :
            vert_list.append([edge.start.real, edge.start.imag])
        vert_arr = np.array(vert_list)
        # print(vert_arr)
        xmin = np.min(vert_arr[:, 0])
        xmax = np.max(vert_arr[:, 0])
        ymin = np.min(vert_arr[:, 1])
        ymax = np.max(vert_arr[:, 1])
        return xmin, ymin, xmax, ymax
    
    def get_center(self) :
        xmin, ymin, xmax, ymax = self.bbox()
        return np.array([(xmin + xmax) / 2, (ymin + ymax) / 2])
    
    def __repr__(self):
        return f"Panel(svg_path={self.svg_path})"

    def approximate_quadratic_bezier_with_cubic_bezier(self, VIS : bool = False) :
        for i, edge in enumerate(self.svg_path):
            if isinstance(edge, svgpath.QuadraticBezier):
                # Convert quadratic to cubic using the standard formula:
                # CP1 = start + 2/3 * (control - start)
                # CP2 = end + 2/3 * (control - end)
                start = edge.start
                end = edge.end
                control = edge.control
                
                cp1 = start + (control - start) * (2/3)
                cp2 = end + (control - end) * (2/3)
                
                self.svg_path[i] = svgpath.CubicBezier(start, cp1, cp2, end)
                
                if VIS :
                    t_list = np.linspace(0, 1, 100)
                    plt.figure(figsize=(10, 10))
                    plt.title("quadratic => cubic bezier")
                    plt.plot(
                        list(map(lambda t : edge.point(t).real, t_list)),
                        list(map(lambda t : edge.point(t).imag, t_list)),
                        color = "black"
                    )
                    plt.plot(
                        list(map(lambda t : self.svg_path[i].point(t).real, t_list)),
                        list(map(lambda t : self.svg_path[i].point(t).imag, t_list)),
                        color = "red"
                    )
                    plt.axis("equal")
                    plt.show()
        return self
    
    def approximate_arc_with_cubic_bezier(self, VIS : bool = False) :
        for i, edge in enumerate(self.svg_path):
            if isinstance(edge, svgpath.Arc):
                x1 = edge.start.real
                y1 = edge.start.imag
                x2 = edge.end.real
                y2 = edge.end.imag
                rx = edge.radius.real
                ry = edge.radius.imag
                rotation = edge.rotation
                large_arc = edge.large_arc
                sweep = edge.sweep
                
                phi_rad = math.radians(rotation)
                cx = (x1 - x2) / 2.0
                cy = (y1 - y2) / 2.0
                
                            
                # x1', y1'
                x1p = math.cos(phi_rad)*cx + math.sin(phi_rad)*cy
                y1p = -math.sin(phi_rad)*cx + math.cos(phi_rad)*cy
                
                # Step 2: rx,ry 스케일링 체크 (여기서는 필요 없으리라 가정)
                lam = (x1p**2)/(rx**2) + (y1p**2)/(ry**2)
                if lam > 1:
                    scale = math.sqrt(lam)
                    rx *= scale
                    ry *= scale

                # Step 3: 계수 (SVG 사양에 따른)
                num = rx**2 * ry**2 - rx**2 * y1p**2 - ry**2 * x1p**2
                den = rx**2 * y1p**2 + ry**2 * x1p**2
                # 부동소수점 오차로 음수 방지
                factor = math.sqrt(max(0, num/den)) if den != 0 else 0
                # SVG 사양에 따라, large_arc와 sweep 플래그가 같으면 부호 반전

                if large_arc == sweep:
                    factor = -factor
                    
                    
                cxp = factor * (rx * y1p / ry)
                cyp = factor * (-ry * x1p / rx)

                # Step 4: 원래 좌표계로 복원 (phi=0이면 회전 없이 중간점에 더함)
                mid_x = (x1 + x2) / 2.0
                mid_y = (y1 + y2) / 2.0
                cx = math.cos(phi_rad)*cxp - math.sin(phi_rad)*cyp + mid_x
                cy = math.sin(phi_rad)*cxp + math.cos(phi_rad)*cyp + mid_y

                # Step 5: 시작각과 끝각 (원 중심 기준)
                theta_start = math.atan2(y1 - cy, x1 - cx)
                theta_end   = math.atan2(y2 - cy, x2 - cx)
                
                # Step 6: 아크의 진행각 delta_theta 결정 (SVG 사양)
                delta_theta = theta_end - theta_start
                
                if sweep:
                    if delta_theta < 0:
                        delta_theta += 2*math.pi
                else:
                    if delta_theta > 0:
                        delta_theta -= 2*math.pi
        
                P0 = (cx + rx * math.cos(theta_start), cy + ry * math.sin(theta_start))
                P3 = (cx + rx * math.cos(theta_end),   cy + ry * math.sin(theta_end))
        
                # Cubic Bézier 공식: k = (4/3)*tan(delta_theta/4)
                k = (4.0/3.0) * math.tan(delta_theta/4.0)
                
                # 컨트롤 포인트
                P1 = (P0[0] - k * rx * math.sin(theta_start),
                    P0[1] + k * ry * math.cos(theta_start))
                P2 = (P3[0] + k * rx * math.sin(theta_end),
                    P3[1] - k * ry * math.cos(theta_end))
                # print(P0, P1, P2, P3)
                
                self.svg_path[i] = svgpath.CubicBezier(
                    P0[0] + P0[1] * 1j, P1[0] + P1[1] * 1j, P2[0] + P2[1] * 1j, P3[0] + P3[1] * 1j
                )
                
                if VIS :
                    t_list = np.linspace(0, 1, 100)
                    plt.figure(figsize=(10, 10))
                    plt.title("arc => cubic bezier")
                    plt.plot(
                        list(map(lambda t : edge.point(t).real, t_list)),
                        list(map(lambda t : edge.point(t).imag, t_list)),
                        color = "black"
                    )
                    plt.plot(
                        list(map(lambda t : self.svg_path[i].point(t).real, t_list)),
                        list(map(lambda t : self.svg_path[i].point(t).imag, t_list)),
                        color = "red"
                    )
                    plt.axis("equal")
                    plt.show()

class SewingPattern :
    def __init__(self,
        panel_svg_path_dict : Dict[str, List[svgpath.Path]],
        stitch_dict : Dict[int, List[Dict]],
        panel_name_refine_map : Dict[str, str] = None,
    ) :
        self.panel_dict = {
            panel_name : SVGPanel(panel_svg_path[0])
            for panel_name, panel_svg_path in panel_svg_path_dict.items()
        }
        self.stitch_dict = StitchDict(stitch_dict)
        self.panel_name_refine_map = panel_name_refine_map
    
    def apply_panel_name_refine_map(
        self, panel_name_refine_map : Dict[str, str] = None
    ) :
        if panel_name_refine_map is None :
            panel_name_refine_map = self.panel_name_refine_map
        for panel_name in self.panel_name_list :
            self.panel_dict[panel_name_refine_map[panel_name]] = self.panel_dict[panel_name]
            del self.panel_dict[panel_name]
    @property
    def panel_name_list(self) :
        return list(self.panel_dict.keys())
    @property
    def panel_list(self) :
        return list(self.panel_dict.values())
    
    def set_panel_start(self, panel_name : str, start_idx : int) :
        panel_edge_count = len(self.panel_dict[panel_name].svg_path)
        panel_edge_idx_map = {
            idx : (idx - start_idx) % panel_edge_count
            for idx in range(panel_edge_count)
        }
        for stch_id, stitch in self.stitch_dict.items() :
            if stitch.panel_0 == panel_name :
                stitch.edge_0 = panel_edge_idx_map[stitch.edge_0]
            if stitch.panel_1 == panel_name :
                stitch.edge_1 = panel_edge_idx_map[stitch.edge_1]
        
        self.panel_dict[panel_name].set_start(start_idx)
        
    
    def reverse_panel_path(self, panel_name : str) :
        panel_edge_len = len(self.panel_dict[panel_name].svg_path)
        self.panel_dict[panel_name].reverse_path()
        for stch_id, stitch in self.stitch_dict.items() :
            if stitch.panel_0 == panel_name :
                stitch.edge_0 = panel_edge_len - 1 - stitch.edge_0
            if stitch.panel_1 == panel_name :
                stitch.edge_1 = panel_edge_len - 1 - stitch.edge_1
       
    def mirror_panel_horizontally(self, panel_name : str) :
        self.panel_dict[panel_name].mirror_horizontal()

    # def mirror_back_panel_horizontally(self) :
    #     for panel_name, panel in self.panel_dict.items() :
    #         if (
    #             panel_name in self.panel_name_refine_map
    #         ) and (
    #             "back" in self.panel_name_refine_map[panel_name]
    #         ) :
    #             panel.mirror_horizontal()
       
    # def unifiy_loop_direction(self, clockwise_only : bool = True) :
    #     for panel_name, panel in self.panel_dict.items() :
    #         if panel.is_clockwise() != clockwise_only :
    #             self.reverse_panel_path(panel_name)
       
       
    def simplyfy_pattern(self) :
        pass
       
    def draw(
        self,
        # ax : plt.Axes = None,
        FIGLEN : int = 5,
        N_SAMPLE_PER_EDGE : int = 80, 
        show=False
    ) :
        NROWS = int(np.ceil(len(self.panel_dict) ** 0.5))
        NCOLS = int(np.ceil(len(self.panel_dict) / NROWS))
        plt.figure(figsize=(FIGLEN*NCOLS, FIGLEN*NROWS))
        
        color_list = plt.cm.rainbow(np.linspace(0, 1, len(self.stitch_dict)))
        for panel_idx, (panel_name, panel) in enumerate(self.panel_dict.items()) :
            stitch_idx_list = []
            edge_color_list = []
            for edge_idx, edge in enumerate(panel.svg_path) :
                stitch_idx = -1
                for stitch_id, stitch in self.stitch_dict.items() :
                    if (
                        stitch.panel_0 == panel_name and stitch.edge_0 == edge_idx
                    ) or (
                        stitch.panel_1 == panel_name and stitch.edge_1 == edge_idx
                    ):
                        stitch_idx = stitch_id
                        break
                stitch_idx_list.append(stitch_idx)
                edge_color_list.append(color_list[stitch_idx] if stitch_idx != -1 else "black")
            
            # print(len(panel.svg_path), len(edge_color_list), len(stitch_idx_list))
            
            
            ax = plt.subplot(NROWS, NCOLS, panel_idx + 1)
            panel.draw(
                ax, panel_name, N_SAMPLE_PER_EDGE,
                stitch_idx_list, edge_color_list
            )
        if show :
            plt.show()
            
    def get_panel_stch_idx_list(self, panel_name : str) :
        panel_stch_idx_list = []
        for edge_idx in range(len(self.panel_dict[panel_name].svg_path)) :
            stitch_idx = -1
            for stch_idx, stitch in self.stitch_dict.items() :
                if (
                    stitch.panel_0 == panel_name and stitch.edge_0 == edge_idx
                ) or (
                    stitch.panel_1 == panel_name and stitch.edge_1 == edge_idx
                ):
                    stitch_idx = stch_idx
                    break
            panel_stch_idx_list.append(stitch_idx)
        return panel_stch_idx_list
    
    def write(self, path : str) :
        pass
    
@dataclass
class ParameterizedSeamLine :
    stch_idx : int = None
    
    whole_stch_vert_idx_arr : np.ndarray = None
    whole_stch_vert_vis_mask : np.ndarray = None
    # whole_stch_vert_projected_pos_arr : np.ndarray = None
    
    segment_vert_idx_arr_list : List[np.ndarray] = None
    segment_vert_pos_arr_list : List[np.ndarray] = None
    segment_edge_len_arr_list : List[np.ndarray] = None
    segment_t_arr_list : List[np.ndarray] = None
    segment_u_arr_list : List[np.ndarray] = None
    segment_v_arr_list : List[np.ndarray] = None
    
    def translate(self, dx : float, dy : float) :
        for segment_vert_pos_arr in self.segment_vert_pos_arr_list :
            segment_vert_pos_arr[:, 0] += dx
            segment_vert_pos_arr[:, 1] += dy

    def reverse_order(self) :
        self.whole_stch_vert_idx_arr = self.whole_stch_vert_idx_arr[::-1]
        self.whole_stch_vert_vis_mask = self.whole_stch_vert_vis_mask[::-1]
        self.segment_vert_idx_arr_list = [
            segment_vert_idx_arr[::-1] for segment_vert_idx_arr in self.segment_vert_idx_arr_list
        ]
        self.segment_vert_pos_arr_list = [
            segment_vert_pos_arr[::-1] for segment_vert_pos_arr in self.segment_vert_pos_arr_list
        ]
        self.segment_edge_len_arr_list = [
            segment_edge_len_arr[::-1] for segment_edge_len_arr in self.segment_edge_len_arr_list
        ]
        self.segment_t_arr_list = [
            1 - segment_t_arr[::-1] for segment_t_arr in self.segment_t_arr_list
        ]
        self.segment_u_arr_list = [
            1 - segment_u_arr[::-1] for segment_u_arr in self.segment_u_arr_list
        ]
        self.segment_v_arr_list = [
            -segment_v_arr[::-1] for segment_v_arr in self.segment_v_arr_list
        ]
    
    def reorder_segments(
        self, order_f
    ) :
        for segment_idx in range(len(self.segment_vert_pos_arr_list)) :
            if not order_f(self.segment_vert_pos_arr_list[segment_idx]) :
                self.segment_vert_pos_arr_list[segment_idx] = self.segment_vert_pos_arr_list[segment_idx][::-1]
                self.segment_edge_len_arr_list[segment_idx] = self.segment_edge_len_arr_list[segment_idx][::-1]
                self.segment_t_arr_list[segment_idx] = 1 - self.segment_t_arr_list[segment_idx][::-1]
                self.segment_u_arr_list[segment_idx] = 1 - self.segment_u_arr_list[segment_idx][::-1]
                self.segment_v_arr_list[segment_idx] = -self.segment_v_arr_list[segment_idx][::-1]

@dataclass
class ParameterizedEdgeLine :
    pass
    
class SingleViewLabel :
    """
    class for label of single image
    """    
    def __init__(self,
        img,
        img_foreground_mask : np.ndarray,
        vert_visibility_mask : np.ndarray,
        vert_projected_pos_arr : np.ndarray,
        fltrd_vis_seam_line_dict : Dict[int, Dict[str, ParameterizedSeamLine]],
    ) :
        self.img = img.convert("RGB")
        self.img_foreground_mask = img_foreground_mask
        self.vert_visibility_mask = vert_visibility_mask
        self.vert_projected_pos_arr = vert_projected_pos_arr
        
        self.seam_line_dict = {}
        for seam_line_idx, seam_line in fltrd_vis_seam_line_dict.items() :
            self.seam_line_dict[seam_line_idx] = ParameterizedSeamLine(
                stch_idx = seam_line_idx,
                whole_stch_vert_idx_arr = seam_line["raw_idx_arr"],
                whole_stch_vert_vis_mask = seam_line["raw_vis_mask"],
                segment_vert_idx_arr_list = seam_line["segment_idx_arr_list"],
                segment_vert_pos_arr_list = seam_line["segment_pos_arr_list"],
                segment_edge_len_arr_list = seam_line["segment_edge_len_arr_list"],
                segment_t_arr_list = seam_line["segment_t_arr_list"],
                segment_u_arr_list = seam_line["segment_u_arr_list"],
                segment_v_arr_list = seam_line["segment_v_arr_list"],
            )

    def crop(self, crop_l : int, crop_t : int, crop_r : int, crop_b : int) :
        """
        crop_l, crop_t, crop_r, crop_b : int
        """
        self.img = self.img.crop((crop_l, crop_t, crop_r, crop_b))
        self.img_foreground_mask = self.img_foreground_mask[crop_t:crop_b, crop_l:crop_r].copy()
        self.vert_projected_pos_arr[:, 0] -= crop_l
        self.vert_projected_pos_arr[:, 1] -= crop_t
        
        for seam_line_idx, seam_line_info in self.seam_line_dict.items() :
            seam_line_info.translate(-crop_l, -crop_t)

    def pad(
        self, pad_l : int, pad_t : int, pad_r : int, pad_b : int) :
        """
        pad the image and the foreground mask
        """
        self.img = Image.fromarray(np.pad(
            np.array(self.img),
            pad_width=((pad_t, pad_b), (pad_l, pad_r), (0, 0)),
            mode="constant",
            constant_values=255
        ))
        self.img_foreground_mask = np.pad(
            self.img_foreground_mask,
            pad_width=((pad_t, pad_b), (pad_l, pad_r)),
            mode="constant",
            constant_values=0
        )
        
        for seam_line_idx, seam_line_info in self.seam_line_dict.items() : 
            seam_line_info.translate(pad_l, pad_t)
    
    def translate(self, dx : float, dy : float) :
        for seam_line_idx, seam_line_info in self.seam_line_dict.items() :
            seam_line_info.translate(dx, dy)

class UnconstrainedFewViewLabel :
    def __init__(self,
        sewing_pattern : SewingPattern,
        vert_visibility_mask_list : List[np.ndarray],
    ) :
        self.sewing_pattern = sewing_pattern
        self.img = None
        self.seam_line_dict_list = []
        self.vert_visibility_mask_list = vert_visibility_mask_list
        
    def order_seam(self) :
        pass

    def mirror_back_panel_horizontally(self) :
        for panel_name, panel in self.sewing_pattern.panel_dict.items() :
            if (
                panel_name in self.sewing_pattern.panel_name_refine_map
            ) and (
                "back" in self.sewing_pattern.panel_name_refine_map[panel_name]
            ) :
                panel.mirror_horizontal()
    
    def unify_loop_direction(self, clockwise_only : bool = True) :
        for panel_name, panel in self.sewing_pattern.panel_dict.items() :
            if panel.is_clockwise() != clockwise_only :
                self.sewing_pattern.reverse_panel_path(panel_name)

    # def normalize_coord(self, width : int, height : int, resize_img : bool = False) :
    #     if resize_img :
    #         self.img = self.img.resize((width, height))
            


def get_poc_dataset_view_name_list() :
    return ["front", "back", "left", "right"]


def get_bbox_from_mask(img_mask) :
    """
    (x1, y1, x2, y2)
    """
    fg_ycoord, fg_xcoord = np.where(img_mask > 0)
    fg_ycoord_min = np.min(fg_ycoord)
    fg_ycoord_max = np.max(fg_ycoord)
    fg_xcoord_min = np.min(fg_xcoord)
    fg_xcoord_max = np.max(fg_xcoord)
    return (fg_xcoord_min, fg_ycoord_min, fg_xcoord_max, fg_ycoord_max)

def read_poc_files(
    garment_path: str,
    return_data_list = [
        "rendered_image_dict",
        "panel_svg_path_dict",
        "stitch_dict",
        "panel_vertex_mask_dict"
        "vertex_visibility_mask_dict",
        "projected_vertex_pose_dict",
        "fltrd_vis_seam_line_dict",
        "box_mesh",
    ]
) :
    garment_id = os.path.basename(garment_path)

    SPEC_FILE_PATH = os.path.join(garment_path, f"{garment_id}_specification.json")
    pattern = pyg.pattern.wrappers.VisPattern(SPEC_FILE_PATH)

    if "rendered_image_dict" in return_data_list :
        rendered_image_dict = {}
        for side in ["front", "back", "left", "right"] :
            rendered_image_dict[side] = Image.open(os.path.join(garment_path, f"rendered_{side}.png"))
    
    if "panel_svg_path_dict" in return_data_list :
        # Get Garment Blueprint
        panel_svg_path_dict = {
            panel_name : pattern._draw_a_panel(
                panel_name, apply_transform=False, fill=True
            )
            for panel_name in pattern.panel_order()
        }
    if "stitch_dict" in return_data_list :
        stitch_dict = {
            i : v for i, v in enumerate(pattern.pattern['stitches'])
        }
    if "panel_vertex_mask_dict" in return_data_list :
        with open(os.path.join(garment_path, f"panel_vertex_mask_dict.pkl"), "rb") as f :
            panel_vertex_mask_dict = pickle.load(f)            
    if "vertex_visibility_mask_dict" in return_data_list :
        with open(os.path.join(garment_path, f"{garment_id}_vertex_visibility_mask.pkl"), "rb") as f :
            vertex_visibility_mask_dict = pickle.load(f)            
    if "projected_vertex_pose_dict" in return_data_list :
        with open(os.path.join(garment_path, f"{garment_id}_projected_vertex_pose.pkl"), "rb") as f :
            projected_vertex_pose_dict = pickle.load(f)
    if "fltrd_vis_seam_line_dict" in return_data_list :
        with open(os.path.join(garment_path, f"{garment_id}_fltrd_vis_seam_line_dict.pkl"), "rb") as f :
            fltrd_vis_seam_line_dict = pickle.load(f)
    if "box_mesh" in return_data_list :
        box_mesh = trimesh.load_mesh(
            os.path.join(garment_path, f"{garment_id}_boxmesh.ply"),
            process=False
        )
    return [values for key, values in locals().items() if key in return_data_list]


def read_poc_datapoint(
    garment_path, 
    view_name_list = ["front", "back", "left", "right"],
    panel_name_refine_map = None,
    return_data_list = [
        "rendered_image_dict",
        "panel_svg_path_dict",
        "stitch_dict",
        "panel_vertex_mask_dict",
        "vertex_visibility_mask_dict",
        "projected_vertex_pose_dict",
        "fltrd_vis_seam_line_dict",
        # "box_mesh",
    ]
) :
    (
        rendered_image_dict,
        panel_svg_path_dict,
        stitch_dict,
        panel_vertex_mask_dict,
        vertex_visibility_mask_dict,
        projected_vertex_pose_dict,
        fltrd_vis_seam_line_dict,
        # box_mesh,
    ) = read_poc_files(
        garment_path,
        # os.path.join(DATASET_ROOT, garment_path),
        return_data_list,
    )
    view_label_dict = {}
    for side in view_name_list :
        view_label_dict[side] = SingleViewLabel(
            img = rendered_image_dict[side],
            img_foreground_mask = np.array(rendered_image_dict[side].getchannel("A")),
            vert_visibility_mask = vertex_visibility_mask_dict[side],
            vert_projected_pos_arr = projected_vertex_pose_dict[side],
            fltrd_vis_seam_line_dict = fltrd_vis_seam_line_dict[side],
        )
    
    return view_label_dict, SewingPattern(
        panel_svg_path_dict, stitch_dict, panel_name_refine_map
    ), (
        # box_mesh,
        panel_vertex_mask_dict
    )
    

# Record distance of Proximate points of panels

In [68]:
GARMENT_METADATA_PATH = "garment_metadata_with_height.csv"

garment_metadata_df = pd.read_csv(GARMENT_METADATA_PATH)
garment_metadata_df

,garment_path,split,garment_id,max_token_length,drape_failed,has_digit_in_panel_name,panel_count,garment_type,HOOD,LARGE_ARC,max_height,min_height,nearest_dist
0,garments_5000_0/default_body/rand_RVS1QWPXD0,test,rand_RVS1QWPXD0,815,False,False,14,OUTFIT,False,True,146.500946,56.229393,1.774729
1,garments_5000_0/default_body/rand_M4Z0YOEJ6V,test,rand_M4Z0YOEJ6V,1123,False,False,16,OUTFIT,False,False,146.578644,16.866879,1.774729
2,garments_5000_0/default_body/rand_E489IFV849,test,rand_E489IFV849,774,False,False,20,OUTFIT,False,True,143.456879,51.288692,0.787160
3,garments_5000_0/default_body/rand_TNYM41T44B,test,rand_TNYM41T44B,484,False,False,8,TOP,False,False,149.193817,108.870781,1.774729
4,garments_5000_0/default_body/rand_H1HW5UW5S7,test,rand_H1HW5UW5S7,1100,False,False,22,TOP,False,True,148.551376,101.899254,0.382277
...,...,...,...,...,...,...,...,...,...,...,...,...,...
115190,garments_5000_35/default_body/rand_U7D7P03PKC,valid,rand_U7D7P03PKC,830,False,True,14,OUTFIT,False,True,143.606247,1.800807,0.253538
115191,garments_5000_35/default_body/rand_FHFVUV6ZZI,valid,rand_FHFVUV6ZZI,474,False,True,9,OUTFIT,False,False,145.993484,-5.595509,2.799091
115192,garments_5000_35/default_body/rand_4MCUZ6H8DY,valid,rand_4MCUZ6H8DY,341,False,False,4,TOP,False,False,149.126175,108.542320,1.774729
115193,garments_5000_35/default_body/rand_7748RZT21S,valid,rand_7748RZT21S,1057,False,True,12,OUTFIT,False,False,145.542572,26.069792,1.774729


In [69]:
garment_metadata_df

,garment_path,split,garment_id,max_token_length,drape_failed,has_digit_in_panel_name,panel_count,garment_type,HOOD,LARGE_ARC,max_height,min_height,nearest_dist
0,garments_5000_0/default_body/rand_RVS1QWPXD0,test,rand_RVS1QWPXD0,815,False,False,14,OUTFIT,False,True,146.500946,56.229393,1.774729
1,garments_5000_0/default_body/rand_M4Z0YOEJ6V,test,rand_M4Z0YOEJ6V,1123,False,False,16,OUTFIT,False,False,146.578644,16.866879,1.774729
2,garments_5000_0/default_body/rand_E489IFV849,test,rand_E489IFV849,774,False,False,20,OUTFIT,False,True,143.456879,51.288692,0.787160
3,garments_5000_0/default_body/rand_TNYM41T44B,test,rand_TNYM41T44B,484,False,False,8,TOP,False,False,149.193817,108.870781,1.774729
4,garments_5000_0/default_body/rand_H1HW5UW5S7,test,rand_H1HW5UW5S7,1100,False,False,22,TOP,False,True,148.551376,101.899254,0.382277
...,...,...,...,...,...,...,...,...,...,...,...,...,...
115190,garments_5000_35/default_body/rand_U7D7P03PKC,valid,rand_U7D7P03PKC,830,False,True,14,OUTFIT,False,True,143.606247,1.800807,0.253538
115191,garments_5000_35/default_body/rand_FHFVUV6ZZI,valid,rand_FHFVUV6ZZI,474,False,True,9,OUTFIT,False,False,145.993484,-5.595509,2.799091
115192,garments_5000_35/default_body/rand_4MCUZ6H8DY,valid,rand_4MCUZ6H8DY,341,False,False,4,TOP,False,False,149.126175,108.542320,1.774729
115193,garments_5000_35/default_body/rand_7748RZT21S,valid,rand_7748RZT21S,1057,False,True,12,OUTFIT,False,False,145.542572,26.069792,1.774729


In [70]:
_ = '''
PROXIMATE_DIST_LOG_PATH = "proximate_dist_log.txt"

from tqdm import tqdm

with open(PROXIMATE_DIST_LOG_PATH, "a") as f :
    for idx, garment in tqdm(garment_metadata_df.iterrows()) :
        
        garment_base_path = garment["garment_path"]
        garment_id = garment["garment_id"]
        garment_type = garment["garment_type"]
        
        garment_path = os.path.join(
            DATASET_ROOT, "GarmentCodeData_v2",
            garment_base_path,
        )
        
        panel_svg_path_dict = read_poc_files(
            garment_path,
            ["panel_svg_path_dict"]
        )[0]
        

        panel_list = list(map(
            lambda x : SVGPanel(x[0]),
            panel_svg_path_dict.values()
        ))
        
        final_min_dist = np.inf
        for panel in panel_list :
            min_dist = panel.find_narrowest_distance()   
            final_min_dist = np.min([final_min_dist, min_dist])
        
        garment_metadata_df.loc[idx, "nearest_dist"] = final_min_dist
            
        f.write(f"{garment_base_path},{final_min_dist}\n")


garment_metadata_df.to_csv(GARMENT_METADATA_PATH, index=False)        
'''

In [71]:
garment_metadata_df

,garment_path,split,garment_id,max_token_length,drape_failed,has_digit_in_panel_name,panel_count,garment_type,HOOD,LARGE_ARC,max_height,min_height,nearest_dist
0,garments_5000_0/default_body/rand_RVS1QWPXD0,test,rand_RVS1QWPXD0,815,False,False,14,OUTFIT,False,True,146.500946,56.229393,1.774729
1,garments_5000_0/default_body/rand_M4Z0YOEJ6V,test,rand_M4Z0YOEJ6V,1123,False,False,16,OUTFIT,False,False,146.578644,16.866879,1.774729
2,garments_5000_0/default_body/rand_E489IFV849,test,rand_E489IFV849,774,False,False,20,OUTFIT,False,True,143.456879,51.288692,0.787160
3,garments_5000_0/default_body/rand_TNYM41T44B,test,rand_TNYM41T44B,484,False,False,8,TOP,False,False,149.193817,108.870781,1.774729
4,garments_5000_0/default_body/rand_H1HW5UW5S7,test,rand_H1HW5UW5S7,1100,False,False,22,TOP,False,True,148.551376,101.899254,0.382277
...,...,...,...,...,...,...,...,...,...,...,...,...,...
115190,garments_5000_35/default_body/rand_U7D7P03PKC,valid,rand_U7D7P03PKC,830,False,True,14,OUTFIT,False,True,143.606247,1.800807,0.253538
115191,garments_5000_35/default_body/rand_FHFVUV6ZZI,valid,rand_FHFVUV6ZZI,474,False,True,9,OUTFIT,False,False,145.993484,-5.595509,2.799091
115192,garments_5000_35/default_body/rand_4MCUZ6H8DY,valid,rand_4MCUZ6H8DY,341,False,False,4,TOP,False,False,149.126175,108.542320,1.774729
115193,garments_5000_35/default_body/rand_7748RZT21S,valid,rand_7748RZT21S,1057,False,True,12,OUTFIT,False,False,145.542572,26.069792,1.774729


In [89]:
x = 1
f"xx{x:02d}"

'xx01'

## Record Panel Combination

In [72]:
import pandas as pd
import json
from tqdm import tqdm

GARMENT_METADATA_PATH = "garment_metadata_with_height.csv"
garment_metadata_df = pd.read_csv(GARMENT_METADATA_PATH)

PANEL_COMBINATION_LOG_PATH = "panel_combination_log.txt"

combination_set = set()

with open(PANEL_COMBINATION_LOG_PATH, "a") as f :
    for idx, garment in tqdm(garment_metadata_df.iterrows()) :
        
        garment_base_path = garment["garment_path"]
        garment_id = garment["garment_id"]
        garment_type = garment["garment_type"]
        
        garment_path = os.path.join(
            DATASET_ROOT, "GarmentCodeData_v2",
            garment_base_path,
        )
        
        SPEC_FILE_PATH = os.path.join(garment_path, f"{garment_id}_specification.json")
        pattern = pyg.pattern.wrappers.VisPattern(SPEC_FILE_PATH)
        
        panel_name_list = pattern.panel_order()
        
        combination_set.add(tuple(panel_name_list))
        
        

115195it [00:38, 2958.05it/s]


## Edit GCD json Directly
- remove intricate pants/sleeve cuff/skirt
- remove hood

In [73]:
from pathlib import Path

delete_panel_name_list = [
    'left_collar_back',
    'left_collar_front',
    'right_collar_back',
    'right_collar_front',
    'left_hood',
    'right_hood',
    'pant_l_cuff_b',
    'pant_l_cuff_f',
    'pant_l_cuff_skirt_b',
    'pant_l_cuff_skirt_f',
    'pant_r_cuff_b',
    'pant_r_cuff_f',
    'pant_r_cuff_skirt_b',
    'pant_r_cuff_skirt_f',
    'sl_left_cuff_b',
    'sl_left_cuff_f',
    'sl_left_cuff_skirt_b',
    'sl_left_cuff_skirt_f',
    'sl_right_cuff_b',
    'sl_right_cuff_f',
    'sl_right_cuff_skirt_b',
    'sl_right_cuff_skirt_f',
]

IDX = 0
for IDX in tqdm(range(len(garment_metadata_df))) :
    garment_base_path = garment_metadata_df.loc[IDX, "garment_path"]

    garment_split, _, garment_id = list(Path(garment_base_path).parts)[-3:]
        

    garment_path = os.path.join(
        DATASET_ROOT, "GarmentCodeData_v2",
        garment_base_path,
    )

    # ===============================
    SPEC_FILE_PATH = os.path.join(
        garment_path, f"{garment_id}_specification.json"
    )
    EDITED_SPEC_FILE_PATH = os.path.join(
        garment_path, f"{garment_id}_specification__01.json"
    )
    # ===============================

    with open(SPEC_FILE_PATH, "r") as f :
        spec_dict = json.load(f)

    new_panels = {}
    for panel_name in spec_dict['pattern']['panels'].keys() :
        if panel_name not in delete_panel_name_list :
            new_panels[panel_name] = spec_dict['pattern']['panels'][panel_name]
    spec_dict['pattern']['panels'] = new_panels

    new_stitches = []
    for stitch in spec_dict['pattern']['stitches'] :
        if (
            stitch[0]['panel'] not in delete_panel_name_list
        ) and (
            stitch[1]['panel'] not in delete_panel_name_list
        ) :
            new_stitches.append(stitch)
    spec_dict['pattern']['stitches'] = new_stitches

    spec_dict['pattern']['panel_order'] = list(filter(
        lambda x : x not in delete_panel_name_list,
        spec_dict['pattern']['panel_order']
    ))

    with open(EDITED_SPEC_FILE_PATH, "w") as f :
        json.dump(spec_dict, f, indent=4)


  0%|          | 0/115195 [00:00<?, ?it/s]

100%|██████████| 115195/115195 [19:24<00:00, 98.90it/s] 


## Update Metadata for edited garments

In [74]:
gcd_01_metadata_df.columns

Index(['garment_path', 'split', 'garment_id', 'max_token_length',
       'drape_failed', 'has_digit_in_panel_name', 'panel_count',
       'garment_type', 'LARGE_ARC', ' max_height', ' min_height',
       'nearest_dist', 'has_torso', 'has_pants', 'has_skirt', 'class'],
      dtype='object')

In [ ]:
x = 10
f"xx{x:02d}"

In [75]:
gcd_01_metadata_df = garment_metadata_df.copy()
# drop HOOD column
gcd_01_metadata_df = gcd_01_metadata_df.drop(columns=["HOOD"])
# update garment_path
gcd_01_metadata_df["has_torso"] = None
gcd_01_metadata_df["has_pants"] = None
gcd_01_metadata_df["has_skirt"] = None

for idx, row in tqdm(gcd_01_metadata_df.iterrows()) :
    garment_base_path = row["garment_path"]
    
    garment_split, _, garment_id = list(Path(garment_base_path).parts)[-3:]

    garment_path = os.path.join(
        DATASET_ROOT, "GarmentCodeData_v2",
        garment_base_path,
    )

    # ===============================
    # SPEC_FILE_PATH = os.path.join(
    #     garment_path, f"{garment_id}_specification.json"
    # )
    # with open(SPEC_FILE_PATH, "r") as f :
    #     spec_dict = json.load(f)
        
    EDITED_SPEC_FILE_PATH = os.path.join(
        garment_path, f"{garment_id}_specification__01.json"
    )

    with open(EDITED_SPEC_FILE_PATH, "r") as f :
        edited_spec_dict = json.load(f)

    pattern = pyg.pattern.wrappers.VisPattern(EDITED_SPEC_FILE_PATH)
    panel_svg_path_dict = {
        panel_name : pattern._draw_a_panel(
            panel_name, apply_transform=False, fill=True
        )
        for panel_name in pattern.panel_order()
    }
    stitch_dict = {
        i : v for i, v in enumerate(pattern.pattern['stitches'])
    }
    sewing_pattern = SewingPattern(
        panel_svg_path_dict, stitch_dict
    )
    
    panel_name_list = list(panel_svg_path_dict.keys())
    
    gcd_01_metadata_df.loc[idx, "panel_count"] = len(panel_name_list)
    
    has_torso = False
    has_pants = False
    has_skirt = False
    for panel_name in panel_name_list :
        if "torso" in panel_name :
            has_torso = True
        if "pant" in panel_name :
            has_pants = True
        if "skirt" in panel_name :
            has_skirt = True
            
    gcd_01_metadata_df.loc[idx, "has_torso"] = has_torso
    gcd_01_metadata_df.loc[idx, "has_pants"] = has_pants
    gcd_01_metadata_df.loc[idx, "has_skirt"] = has_skirt
    
    # sewing_pattern.draw()
    # plt.show()    
    # print(panel_name_list)
    # print(garment_base_path)
    # print(garment_split)
    # print(garment_id)
    # print(SPEC_FILE_PATH)
    # print(EDITED_SPEC_FILE_PATH)
    # print(spec_dict == edited_spec_dict)
    # print()

    
gcd_01_metadata_df.to_csv(
    "gcd_01_metadata_df.csv",index=False
)
gcd_01_metadata_df

0it [00:00, ?it/s]

115195it [05:32, 346.58it/s]


,garment_path,split,garment_id,max_token_length,drape_failed,has_digit_in_panel_name,panel_count,garment_type,LARGE_ARC,max_height,min_height,nearest_dist,has_torso,has_pants,has_skirt
0,garments_5000_0/default_body/rand_RVS1QWPXD0,test,rand_RVS1QWPXD0,815,False,False,10,OUTFIT,True,146.500946,56.229393,1.774729,True,False,True
1,garments_5000_0/default_body/rand_M4Z0YOEJ6V,test,rand_M4Z0YOEJ6V,1123,False,False,12,OUTFIT,False,146.578644,16.866879,1.774729,True,False,True
2,garments_5000_0/default_body/rand_E489IFV849,test,rand_E489IFV849,774,False,False,12,OUTFIT,True,143.456879,51.288692,0.787160,True,False,True
3,garments_5000_0/default_body/rand_TNYM41T44B,test,rand_TNYM41T44B,484,False,False,4,TOP,False,149.193817,108.870781,1.774729,True,False,False
4,garments_5000_0/default_body/rand_H1HW5UW5S7,test,rand_H1HW5UW5S7,1100,False,False,10,TOP,True,148.551376,101.899254,0.382277,True,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115190,garments_5000_35/default_body/rand_U7D7P03PKC,valid,rand_U7D7P03PKC,830,False,True,14,OUTFIT,True,143.606247,1.800807,0.253538,True,False,True
115191,garments_5000_35/default_body/rand_FHFVUV6ZZI,valid,rand_FHFVUV6ZZI,474,False,True,9,OUTFIT,False,145.993484,-5.595509,2.799091,True,False,True
115192,garments_5000_35/default_body/rand_4MCUZ6H8DY,valid,rand_4MCUZ6H8DY,341,False,False,4,TOP,False,149.126175,108.542320,1.774729,True,False,False
115193,garments_5000_35/default_body/rand_7748RZT21S,valid,rand_7748RZT21S,1057,False,True,12,OUTFIT,False,145.542572,26.069792,1.774729,True,False,True


In [76]:
has_torso_arr = gcd_01_metadata_df.has_torso.to_numpy()
has_pants_arr = gcd_01_metadata_df.has_pants.to_numpy()
has_skirt_arr = gcd_01_metadata_df.has_skirt.to_numpy()

has_torso_arr.sum(), has_pants_arr.sum(), has_skirt_arr.sum()



(77357, 15976, 69907)

In [77]:
for idx, row in tqdm(gcd_01_metadata_df.iterrows()) :
    if row["has_torso"] == False :
        type = "bottom"
    else :
        if row["has_pants"] == True or row["has_skirt"] == True :
            type = "outfit"
        else :
            type = "top"
    gcd_01_metadata_df.loc[idx, "class"] = type


115195it [00:35, 3222.63it/s]


In [78]:
gcd_01_metadata_df.to_csv(
    "gcd_01_metadata_df.csv",index=False
)
gcd_01_metadata_df


,garment_path,split,garment_id,max_token_length,drape_failed,has_digit_in_panel_name,panel_count,garment_type,LARGE_ARC,max_height,min_height,nearest_dist,has_torso,has_pants,has_skirt,class
0,garments_5000_0/default_body/rand_RVS1QWPXD0,test,rand_RVS1QWPXD0,815,False,False,10,OUTFIT,True,146.500946,56.229393,1.774729,True,False,True,outfit
1,garments_5000_0/default_body/rand_M4Z0YOEJ6V,test,rand_M4Z0YOEJ6V,1123,False,False,12,OUTFIT,False,146.578644,16.866879,1.774729,True,False,True,outfit
2,garments_5000_0/default_body/rand_E489IFV849,test,rand_E489IFV849,774,False,False,12,OUTFIT,True,143.456879,51.288692,0.787160,True,False,True,outfit
3,garments_5000_0/default_body/rand_TNYM41T44B,test,rand_TNYM41T44B,484,False,False,4,TOP,False,149.193817,108.870781,1.774729,True,False,False,top
4,garments_5000_0/default_body/rand_H1HW5UW5S7,test,rand_H1HW5UW5S7,1100,False,False,10,TOP,True,148.551376,101.899254,0.382277,True,False,True,outfit
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115190,garments_5000_35/default_body/rand_U7D7P03PKC,valid,rand_U7D7P03PKC,830,False,True,14,OUTFIT,True,143.606247,1.800807,0.253538,True,False,True,outfit
115191,garments_5000_35/default_body/rand_FHFVUV6ZZI,valid,rand_FHFVUV6ZZI,474,False,True,9,OUTFIT,False,145.993484,-5.595509,2.799091,True,False,True,outfit
115192,garments_5000_35/default_body/rand_4MCUZ6H8DY,valid,rand_4MCUZ6H8DY,341,False,False,4,TOP,False,149.126175,108.542320,1.774729,True,False,False,top
115193,garments_5000_35/default_body/rand_7748RZT21S,valid,rand_7748RZT21S,1057,False,True,12,OUTFIT,False,145.542572,26.069792,1.774729,True,False,True,outfit


In [79]:
MIN_DIST_THRESH = 1.5
PANEL_COUNT_THRESH = 12

 
filtered_garment_metadata_df = gcd_01_metadata_df[
    (
        garment_metadata_df["nearest_dist"] > MIN_DIST_THRESH
    ) & (
        garment_metadata_df["panel_count"] <= PANEL_COUNT_THRESH
    ) & (
        garment_metadata_df["LARGE_ARC"] == False
    ) & (
        garment_metadata_df["drape_failed"] == False
    ) 
]


print(
    filtered_garment_metadata_df[
        (
            filtered_garment_metadata_df["class"] == "top"
        ) & (
            filtered_garment_metadata_df["split"] == "train"
        )
    ].__len__()
)

print(
    filtered_garment_metadata_df[
        (
            filtered_garment_metadata_df["class"] == "top"
        ) & (
            filtered_garment_metadata_df["split"] == "valid"
        )
    ].__len__()
)

print(
    filtered_garment_metadata_df[
        (
            filtered_garment_metadata_df["class"] == "top"
        ) & (
            filtered_garment_metadata_df["split"] == "test"
        )
    ].__len__()
)
print("===================")

print(
    filtered_garment_metadata_df[
        (
            filtered_garment_metadata_df["class"] == "bottom"
        ) & (
            filtered_garment_metadata_df["split"] == "train"
        )
    ].__len__()
)

print(
    filtered_garment_metadata_df[
        (
            filtered_garment_metadata_df["class"] == "bottom"
        ) & (
            filtered_garment_metadata_df["split"] == "valid"
        )
    ].__len__()
)


print(
    filtered_garment_metadata_df[
        (
            filtered_garment_metadata_df["class"] == "bottom"
        ) & (
            filtered_garment_metadata_df["split"] == "test"
        )
    ].__len__()
)

print("===================")


print(
    filtered_garment_metadata_df[
        (
            filtered_garment_metadata_df["class"] == "outfit"
        ) & (
            filtered_garment_metadata_df["split"] == "train"
        )
    ].__len__()
)

print(
    filtered_garment_metadata_df[
        (
            filtered_garment_metadata_df["class"] == "outfit"
        ) & (
            filtered_garment_metadata_df["split"] == "valid"
        )
    ].__len__()
)


print(
    filtered_garment_metadata_df[
        (
            filtered_garment_metadata_df["class"] == "outfit"
        ) & (
            filtered_garment_metadata_df["split"] == "test"
        )
    ].__len__()
)

10172
608
642
24088
1461
1447
6575
446
408


In [80]:
outfit_path_list = []
top_bottom_path_list = []

for idx_split in range(36) :
    for train_split in ["train", "valid", "test"] :
        garment_df_1 = filtered_garment_metadata_df[
            (
                filtered_garment_metadata_df["garment_path"].str.contains(f"5000_{idx_split}/")
            ) & (
                filtered_garment_metadata_df["split"].str.contains(train_split)
            )
        ]
        
        outfit_df = garment_df_1[garment_df_1["class"] == "outfit"]
        top_df = garment_df_1[garment_df_1["class"] == "top"].sort_values(
            by=" min_height", ascending=True
        )
        bottom_df = garment_df_1[garment_df_1["class"] == "bottom"].sort_values(
            by=" max_height", ascending=True
        )
        
        # print(idx_split, train_split, "outfit", len(outfit_df))
        # print(idx_split, train_split, "top", len(top_df))
        # print(idx_split, train_split, "bottom", len(bottom_df))
        
        outfit_path_list.extend(outfit_df["garment_path"].tolist())
        for top_row, bottom_row in zip(top_df.iterrows(), bottom_df.iterrows()) :
            top_bottom_path_list.append(
                f"{top_row[1].garment_path},{bottom_row[1].garment_path}"
            )
        
    #     print(len(outfit_path_list))
    #     print(len(top_bottom_path_list))
    #     print("-"*100)
    # print("="*100)

In [81]:
import json
with open("gcd_01_outfit_path_list.json", "w") as f :
    json.dump(outfit_path_list, f, indent=4)
with open("gcd_01_top_bottom_path_list.json", "w") as f :
    json.dump(top_bottom_path_list, f, indent=4)

## Record distance of Proximate points of panels

In [82]:
GARMENT_METADATA_PATH = "garment_metadata_with_height.csv"

garment_metadata_df = pd.read_csv(GARMENT_METADATA_PATH)
garment_metadata_df

,garment_path,split,garment_id,max_token_length,drape_failed,has_digit_in_panel_name,panel_count,garment_type,HOOD,LARGE_ARC,max_height,min_height,nearest_dist
0,garments_5000_0/default_body/rand_RVS1QWPXD0,test,rand_RVS1QWPXD0,815,False,False,14,OUTFIT,False,True,146.500946,56.229393,1.774729
1,garments_5000_0/default_body/rand_M4Z0YOEJ6V,test,rand_M4Z0YOEJ6V,1123,False,False,16,OUTFIT,False,False,146.578644,16.866879,1.774729
2,garments_5000_0/default_body/rand_E489IFV849,test,rand_E489IFV849,774,False,False,20,OUTFIT,False,True,143.456879,51.288692,0.787160
3,garments_5000_0/default_body/rand_TNYM41T44B,test,rand_TNYM41T44B,484,False,False,8,TOP,False,False,149.193817,108.870781,1.774729
4,garments_5000_0/default_body/rand_H1HW5UW5S7,test,rand_H1HW5UW5S7,1100,False,False,22,TOP,False,True,148.551376,101.899254,0.382277
...,...,...,...,...,...,...,...,...,...,...,...,...,...
115190,garments_5000_35/default_body/rand_U7D7P03PKC,valid,rand_U7D7P03PKC,830,False,True,14,OUTFIT,False,True,143.606247,1.800807,0.253538
115191,garments_5000_35/default_body/rand_FHFVUV6ZZI,valid,rand_FHFVUV6ZZI,474,False,True,9,OUTFIT,False,False,145.993484,-5.595509,2.799091
115192,garments_5000_35/default_body/rand_4MCUZ6H8DY,valid,rand_4MCUZ6H8DY,341,False,False,4,TOP,False,False,149.126175,108.542320,1.774729
115193,garments_5000_35/default_body/rand_7748RZT21S,valid,rand_7748RZT21S,1057,False,True,12,OUTFIT,False,False,145.542572,26.069792,1.774729


In [83]:
garment_metadata_df

,garment_path,split,garment_id,max_token_length,drape_failed,has_digit_in_panel_name,panel_count,garment_type,HOOD,LARGE_ARC,max_height,min_height,nearest_dist
0,garments_5000_0/default_body/rand_RVS1QWPXD0,test,rand_RVS1QWPXD0,815,False,False,14,OUTFIT,False,True,146.500946,56.229393,1.774729
1,garments_5000_0/default_body/rand_M4Z0YOEJ6V,test,rand_M4Z0YOEJ6V,1123,False,False,16,OUTFIT,False,False,146.578644,16.866879,1.774729
2,garments_5000_0/default_body/rand_E489IFV849,test,rand_E489IFV849,774,False,False,20,OUTFIT,False,True,143.456879,51.288692,0.787160
3,garments_5000_0/default_body/rand_TNYM41T44B,test,rand_TNYM41T44B,484,False,False,8,TOP,False,False,149.193817,108.870781,1.774729
4,garments_5000_0/default_body/rand_H1HW5UW5S7,test,rand_H1HW5UW5S7,1100,False,False,22,TOP,False,True,148.551376,101.899254,0.382277
...,...,...,...,...,...,...,...,...,...,...,...,...,...
115190,garments_5000_35/default_body/rand_U7D7P03PKC,valid,rand_U7D7P03PKC,830,False,True,14,OUTFIT,False,True,143.606247,1.800807,0.253538
115191,garments_5000_35/default_body/rand_FHFVUV6ZZI,valid,rand_FHFVUV6ZZI,474,False,True,9,OUTFIT,False,False,145.993484,-5.595509,2.799091
115192,garments_5000_35/default_body/rand_4MCUZ6H8DY,valid,rand_4MCUZ6H8DY,341,False,False,4,TOP,False,False,149.126175,108.542320,1.774729
115193,garments_5000_35/default_body/rand_7748RZT21S,valid,rand_7748RZT21S,1057,False,True,12,OUTFIT,False,False,145.542572,26.069792,1.774729


In [84]:
_ = '''
PROXIMATE_DIST_LOG_PATH = "proximate_dist_log.txt"

from tqdm import tqdm

with open(PROXIMATE_DIST_LOG_PATH, "a") as f :
    for idx, garment in tqdm(garment_metadata_df.iterrows()) :
        
        garment_base_path = garment["garment_path"]
        garment_id = garment["garment_id"]
        garment_type = garment["garment_type"]
        
        garment_path = os.path.join(
            DATASET_ROOT, "GarmentCodeData_v2",
            garment_base_path,
        )
        
        panel_svg_path_dict = read_poc_files(
            garment_path,
            ["panel_svg_path_dict"]
        )[0]
        

        panel_list = list(map(
            lambda x : SVGPanel(x[0]),
            panel_svg_path_dict.values()
        ))
        
        final_min_dist = np.inf
        for panel in panel_list :
            min_dist = panel.find_narrowest_distance()   
            final_min_dist = np.min([final_min_dist, min_dist])
        
        garment_metadata_df.loc[idx, "nearest_dist"] = final_min_dist
            
        f.write(f"{garment_base_path},{final_min_dist}\n")


garment_metadata_df.to_csv(GARMENT_METADATA_PATH, index=False)        
'''

In [85]:
garment_metadata_df

,garment_path,split,garment_id,max_token_length,drape_failed,has_digit_in_panel_name,panel_count,garment_type,HOOD,LARGE_ARC,max_height,min_height,nearest_dist
0,garments_5000_0/default_body/rand_RVS1QWPXD0,test,rand_RVS1QWPXD0,815,False,False,14,OUTFIT,False,True,146.500946,56.229393,1.774729
1,garments_5000_0/default_body/rand_M4Z0YOEJ6V,test,rand_M4Z0YOEJ6V,1123,False,False,16,OUTFIT,False,False,146.578644,16.866879,1.774729
2,garments_5000_0/default_body/rand_E489IFV849,test,rand_E489IFV849,774,False,False,20,OUTFIT,False,True,143.456879,51.288692,0.787160
3,garments_5000_0/default_body/rand_TNYM41T44B,test,rand_TNYM41T44B,484,False,False,8,TOP,False,False,149.193817,108.870781,1.774729
4,garments_5000_0/default_body/rand_H1HW5UW5S7,test,rand_H1HW5UW5S7,1100,False,False,22,TOP,False,True,148.551376,101.899254,0.382277
...,...,...,...,...,...,...,...,...,...,...,...,...,...
115190,garments_5000_35/default_body/rand_U7D7P03PKC,valid,rand_U7D7P03PKC,830,False,True,14,OUTFIT,False,True,143.606247,1.800807,0.253538
115191,garments_5000_35/default_body/rand_FHFVUV6ZZI,valid,rand_FHFVUV6ZZI,474,False,True,9,OUTFIT,False,False,145.993484,-5.595509,2.799091
115192,garments_5000_35/default_body/rand_4MCUZ6H8DY,valid,rand_4MCUZ6H8DY,341,False,False,4,TOP,False,False,149.126175,108.542320,1.774729
115193,garments_5000_35/default_body/rand_7748RZT21S,valid,rand_7748RZT21S,1057,False,True,12,OUTFIT,False,False,145.542572,26.069792,1.774729


In [86]:
MIN_DIST_THRESH = 1.5
PANEL_COUNT_THRESH = 12

 
filtered_garment_metadata_df = garment_metadata_df[
    (
        garment_metadata_df["nearest_dist"] > MIN_DIST_THRESH
    ) & (
        garment_metadata_df["panel_count"] <= PANEL_COUNT_THRESH
    ) & (
        garment_metadata_df["HOOD"] == False
    ) & (
        garment_metadata_df["LARGE_ARC"] == False
    ) & (
        garment_metadata_df["drape_failed"] == False
    ) 
]


print(
    filtered_garment_metadata_df[
        (
            filtered_garment_metadata_df["garment_type"] == "TOP"
        ) & (
            filtered_garment_metadata_df["split"] == "train"
        )
    ].__len__()
)

print(
    filtered_garment_metadata_df[
        (
            filtered_garment_metadata_df["garment_type"] == "TOP"
        ) & (
            filtered_garment_metadata_df["split"] == "valid"
        )
    ].__len__()
)

print(
    filtered_garment_metadata_df[
        (
            filtered_garment_metadata_df["garment_type"] == "TOP"
        ) & (
            filtered_garment_metadata_df["split"] == "test"
        )
    ].__len__()
)
print("===================")

print(
    filtered_garment_metadata_df[
        (
            filtered_garment_metadata_df["garment_type"] == "BOTTOM"
        ) & (
            filtered_garment_metadata_df["split"] == "train"
        )
    ].__len__()
)

print(
    filtered_garment_metadata_df[
        (
            filtered_garment_metadata_df["garment_type"] == "BOTTOM"
        ) & (
            filtered_garment_metadata_df["split"] == "valid"
        )
    ].__len__()
)


print(
    filtered_garment_metadata_df[
        (
            filtered_garment_metadata_df["garment_type"] == "BOTTOM"
        ) & (
            filtered_garment_metadata_df["split"] == "test"
        )
    ].__len__()
)

print("===================")


print(
    filtered_garment_metadata_df[
        (
            filtered_garment_metadata_df["garment_type"] == "OUTFIT"
        ) & (
            filtered_garment_metadata_df["split"] == "train"
        )
    ].__len__()
)

print(
    filtered_garment_metadata_df[
        (
            filtered_garment_metadata_df["garment_type"] == "OUTFIT"
        ) & (
            filtered_garment_metadata_df["split"] == "valid"
        )
    ].__len__()
)


print(
    filtered_garment_metadata_df[
        (
            filtered_garment_metadata_df["garment_type"] == "OUTFIT"
        ) & (
            filtered_garment_metadata_df["split"] == "test"
        )
    ].__len__()
)

9925
605
619
22618
1377
1366
5539
382
354


## 단순무식하게 일단 조합하고봄ㅇㅇ

In [87]:
outfit_path_list = []
top_bottom_path_list = []

for idx_split in range(36) :
    for train_split in ["train", "valid", "test"] :
        garment_df_1 = filtered_garment_metadata_df[
            (
                filtered_garment_metadata_df["garment_path"].str.contains(f"5000_{idx_split}/")
            ) & (
                filtered_garment_metadata_df["split"].str.contains(train_split)
            )
        ]
        
        outfit_df = garment_df_1[garment_df_1["class"] == "outfit"]
        top_df = garment_df_1[garment_df_1["class"] == "top"].sort_values(
            by=" min_height", ascending=True
        )
        bottom_df = garment_df_1[garment_df_1["class"] == "bottom"].sort_values(
            by=" max_height", ascending=True
        )
        
        # print(idx_split, train_split, "outfit", len(outfit_df))
        # print(idx_split, train_split, "top", len(top_df))
        # print(idx_split, train_split, "bottom", len(bottom_df))
        
        outfit_path_list.extend(outfit_df["garment_path"].tolist())
        for top_row, bottom_row in zip(top_df.iterrows(), bottom_df.iterrows()) :
            top_bottom_path_list.append(
                f"{top_row[1].garment_path},{bottom_row[1].garment_path}"
            )
        
    #     print(len(outfit_path_list))
    #     print(len(top_bottom_path_list))
    #     print("-"*100)
    # print("="*100)

KeyError: 'class'

In [19]:
len(outfit_path_list), len(top_bottom_path_list)

(6275, 11149)

In [20]:
import json
with open("outfit_path_list.json", "w") as f :
    json.dump(outfit_path_list, f, indent=4)
with open("top_bottom_path_list.json", "w") as f :
    json.dump(top_bottom_path_list, f, indent=4)